**NOTICE:**  
The U.S. Army Corps of Engineers, Risk Management Center (USACE-RMC) makes no guarantees about the results, or appropriateness of outputs, obtained from Numerics.

# 01. Probability Distributions (BestFit-First)

This notebook keeps the full distribution set and uses BestFit model objects as the primary interface.

## MLE + Fallback pattern
For each distribution:
1. Try BestFit `MaximumLikelihood`.
2. Validate the resulting parameters.
3. If MLE is unusable, automatically run a stable fallback fit for plotting/reporting.

This matches the resilience pattern used in notebook `00`.

In [29]:
import pythonnet
pythonnet.load("netfx")

import clr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Ensure helper_functions is importable whether notebook runs from repo root or notebooks/
_repo_root = Path.cwd()
_nb_dir = _repo_root / "notebooks"
if _nb_dir.exists() and str(_nb_dir) not in sys.path:
    sys.path.append(str(_nb_dir))
if str(Path.cwd()) not in sys.path:
    sys.path.append(str(Path.cwd()))


from helper_functions import resolve_bestfit_dll, resolve_numerics_dll, convert_to_donet_array

clr.AddReference(str(resolve_numerics_dll()))
clr.AddReference(str(resolve_bestfit_dll()))

from RMC.BestFit import DataFrame, ExactData
from RMC.BestFit.Model import UnivariateDistribution
from Numerics.Distributions import UnivariateDistributionType
from RMC.BestFit.Estimation import MaximumLikelihood, OptimizationMethod
from scipy.optimize import minimize

## Build BestFit Data and Candidate Models
We build one BestFit `DataFrame` and define the same candidate distributions currently used in this notebook.

In [30]:
# Annual-maxima style dataset
annual_peaks = np.array([
    7420, 8010, 8350, 8680, 8920,
    9150, 9430, 9710, 9980, 10240,
    10610, 10980, 11320, 11740, 12110,
    12680, 13150, 13720, 14590, 15440,
    16680, 18120, 19750, 21430, 23680,
], dtype=float)

df = DataFrame()
for i, q in enumerate(annual_peaks):
    df.ExactSeries.Add(ExactData(1999 + i, float(q)))
df.PlottingParameter = 0.0
df.CalculatePlottingPositions()

x = annual_peaks.copy()
mu = float(np.mean(x))
sd = float(np.std(x, ddof=1))

# Current distribution set
specs = {
    "Normal": (UnivariateDistributionType.Normal, [mu, sd]),
    "LogNormal": (UnivariateDistributionType.LogNormal, [np.log(max(mu, 1.0)), 0.25]),
    "GEV": (UnivariateDistributionType.GeneralizedExtremeValue, [mu, 0.82*sd, 0.10]),
    "Gumbel": (UnivariateDistributionType.Gumbel, [mu, 0.80*sd]),
    "Gamma": (UnivariateDistributionType.GammaDistribution, [6.0, max(1.0, mu/6.0)]),
    "Weibull": (UnivariateDistributionType.Weibull, [2.1, max(1.0, mu)]),
    "LP3": (UnivariateDistributionType.LogPearsonTypeIII, [np.log10(max(mu, 1.0)), 0.18, 0.10]),
}

specs

{'Normal': (<UnivariateDistributionType.Normal: 27>,
  [12635.6, 4374.851502241724]),
 'LogNormal': (<UnivariateDistributionType.LogNormal: 23>,
  [np.float64(9.444273505833715), 0.25]),
 'GEV': (<UnivariateDistributionType.GeneralizedExtremeValue: 11>,
  [12635.6, 3587.3782318382137, 0.1]),
 'Gumbel': (<UnivariateDistributionType.Gumbel: 16>,
  [12635.6, 3499.8812017933797]),
 'Gamma': (<UnivariateDistributionType.GammaDistribution: 9>,
  [6.0, 2105.9333333333334]),
 'Weibull': (<UnivariateDistributionType.Weibull: 41>, [2.1, 12635.6]),
 'LP3': (<UnivariateDistributionType.LogPearsonTypeIII: 24>,
  [np.float64(4.101595869168661), 0.18, 0.1])}

## Fit Each Distribution: MLE First, Stable Fallback If Needed
Fallback optimizer directly minimizes negative log-likelihood computed from the fitted BestFit distribution PDF.

In [31]:
def mle_try(model):
    mle = MaximumLikelihood(model, OptimizationMethod.NelderMead)
    mle.Estimate()
    p = np.array([float(v) for v in mle.BestParameterSet.Values], dtype=float)
    return mle, p


def is_param_vector_usable(model, p):
    p = np.asarray(p, dtype=float)
    if np.any(~np.isfinite(p)):
        return False
    try:
        model.SetParameterValues(convert_to_donet_array(p))
        q10 = float(model.Distribution.InverseCDF(0.90))
        q99 = float(model.Distribution.InverseCDF(0.99))
        return np.isfinite(q10) and np.isfinite(q99) and q99 > q10
    except Exception:
        return False


def fallback_fit(model, seed_params):
    model.SetParameterValues(convert_to_donet_array(seed_params))
    lb = np.array([float(p.LowerBound) for p in model.Parameters], dtype=float)
    ub = np.array([float(p.UpperBound) for p in model.Parameters], dtype=float)

    finite_lb = np.where(np.isfinite(lb), lb, -1e6)
    finite_ub = np.where(np.isfinite(ub), ub, 1e6)

    x0 = np.clip(np.array(seed_params, dtype=float), finite_lb + 1e-8, finite_ub - 1e-8)

    def nll(theta):
        theta = np.asarray(theta, dtype=float)
        if np.any(~np.isfinite(theta)):
            return 1e12
        if np.any(theta < finite_lb) or np.any(theta > finite_ub):
            return 1e12
        try:
            model.SetParameterValues(convert_to_donet_array(theta))
            pdf = np.array([max(1e-300, float(model.Distribution.PDF(float(v)))) for v in x], dtype=float)
            if np.any(~np.isfinite(pdf)):
                return 1e12
            return float(-np.sum(np.log(pdf)))
        except Exception:
            return 1e12

    fit = minimize(nll, x0, method="Nelder-Mead", options={"maxiter": 4000, "xatol": 1e-6, "fatol": 1e-6})
    p_hat = np.array(fit.x, dtype=float)

    # Validate optimized result; if invalid, return clipped seed.
    if np.any(~np.isfinite(p_hat)):
        p_hat = x0.copy()
        fit_success = False
    else:
        p_hat = np.clip(p_hat, finite_lb + 1e-8, finite_ub - 1e-8)
        fit_success = bool(fit.success)

    try:
        model.SetParameterValues(convert_to_donet_array(p_hat))
    except Exception:
        p_hat = x0.copy()
        model.SetParameterValues(convert_to_donet_array(p_hat))
        fit_success = False

    return p_hat, fit_success, float(fit.fun) if np.isfinite(fit.fun) else 1e12


rows = []
fitted_models = {}

for name, (dtype, seed_params) in specs.items():
    model = UnivariateDistribution(df, dtype)

    mle_ok = False
    mle_params = np.array(seed_params, dtype=float)
    try:
        mle, mle_params = mle_try(model)
        mle_ok = is_param_vector_usable(model, mle_params)
    except Exception:
        mle_ok = False

    if mle_ok:
        final_params = mle_params
        fit_method = "MLE"
        fit_success = True
        model.SetParameterValues(convert_to_donet_array(final_params))
    else:
        final_params, fit_success, _ = fallback_fit(model, seed_params)
        fit_method = "Fallback"

    # Robust log-likelihood evaluation with guard
    ll = np.nan
    try:
        ll = float(model.LogLikelihood(convert_to_donet_array(final_params)))
        if (not np.isfinite(ll)) or (ll < -1e300):
            raise ValueError
    except Exception:
        try:
            pdf = np.array([max(1e-300, float(model.Distribution.PDF(float(v)))) for v in x], dtype=float)
            ll = float(np.sum(np.log(pdf)))
        except Exception:
            ll = np.nan

    rows.append({
        "distribution": name,
        "fit_method": fit_method,
        "fit_success": fit_success,
        "mean": float(model.Distribution.Mean),
        "std": float(model.Distribution.StandardDeviation),
        "q90": float(model.Distribution.InverseCDF(0.90)),
        "q99": float(model.Distribution.InverseCDF(0.99)),
        "log_likelihood": ll,
    })

    fitted_models[name] = model

summary = pd.DataFrame(rows)
summary

ArgumentOutOfRangeException: The location parameter ξ (Xi) must be a number.
Parameter name: Xi
   at Numerics.Distributions.GeneralizedExtremeValue.ValidateParameters(Double location, Double scale, Double shape, Boolean throwException)
   at Numerics.Distributions.GeneralizedExtremeValue.PDF(Double x)

In [ ]:
# Compare fitted CDF curves
xg = np.linspace(np.percentile(x, 1), np.percentile(x, 99) * 1.4, 600)

plt.figure(figsize=(9, 5.5))
for name in ["Normal", "LogNormal", "GEV", "Gumbel", "Gamma", "Weibull", "LP3"]:
    m = fitted_models[name]
    y = np.array([float(m.Distribution.CDF(float(v))) for v in xg])
    plt.plot(xg, y, lw=2, label=name)

plt.title("Fitted CDF Comparison (MLE with Fallback)")
plt.xlabel("Peak flow")
plt.ylabel("F(x)")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend(ncol=2)
plt.tight_layout()

# Return-period table from fitted models
T = np.array([2, 5, 10, 25, 50, 100, 200], dtype=float)
F = 1.0 - 1.0/T
rows = []
for t, f in zip(T, F):
    row = {"T": float(t)}
    for name, m in fitted_models.items():
        row[name] = float(m.Distribution.InverseCDF(float(f)))
    rows.append(row)

qtab = pd.DataFrame(rows)
qtab

## Summary and Exercises

All current distributions are retained. Each is now fitted with an MLE-first path plus automatic fallback when MLE is unstable.

Recommended exercises:
1. Add a column showing whether MLE passed or fallback was used.
2. Compare AIC/BIC rankings after fitting.
3. Replace synthetic peaks with observed annual maxima and rerun.